# ICD311 — Deep Learning & Neural Networks Lab
Mulungushi University — School of Engineering and Technology

This notebook walks through every practical task in the lab. Run cells top to bottom.
It imports reusable code from `../src/` so the logic is tested and kept in one place.


## 2. Environment check
Confirm Python, NumPy, PyTorch versions and device.

In [1]:
import sys
import numpy as np
print("Python:", sys.version)
print("NumPy:", np.__version__)

import torch
print("torch.__version__:", torch.__version__)
print("torch.cuda.is_available():", torch.cuda.is_available())
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)


Python: 3.12.5 (tags/v3.12.5:ff3bc82, Aug  6 2024, 20:45:27) [MSC v.1940 64 bit (AMD64)]
NumPy: 2.4.4
torch.__version__: 2.14.0+cpu
torch.cuda.is_available(): False
Using device: cpu


## 3. Warm-Up: Understand the Neuron Before Coding

**A. Manual calculation** — x = [2, 1], w = [0.5, -1], b = 0.2

z = w1*x1 + w2*x2 + b = (0.5)(2) + (-1)(1) + 0.2 = 1 - 1 + 0.2 = **0.2**
ReLU(0.2) = **0.2**

**B. Changed input** — x = [0, 2], same w, b

z = (0.5)(0) + (-1)(2) + 0.2 = -1.8
ReLU(-1.8) = **0** — ReLU clips any negative pre-activation to exactly 0, so a negative z
contributes nothing to the next layer and carries no gradient for that example.

**C. Parameter count for 2 → 3 → 1**

- Layer 1 (2→3): 2×3 weights + 3 biases = 9
- Layer 2 (3→1): 3×1 weights + 1 bias = 4
- **Total = 13**

Verify both by hand and in code below.


In [2]:
# Verify the manual calculations in code
def relu(z):
    return max(0.0, z)

# A
w = [0.5, -1]; b = 0.2
xA = [2, 1]
zA = w[0]*xA[0] + w[1]*xA[1] + b
print("A) z =", zA, " ReLU(z) =", relu(zA))

# B
xB = [0, 2]
zB = w[0]*xB[0] + w[1]*xB[1] + b
print("B) z =", zB, " ReLU(z) =", relu(zB))

# C — parameter count for a dense 2 -> 3 -> 1 network
params_layer1 = 2*3 + 3
params_layer2 = 3*1 + 1
print("C) total trainable parameters =", params_layer1 + params_layer2)


A) z = 0.2  ReLU(z) = 0.2
B) z = -1.8  ReLU(z) = 0.0
C) total trainable parameters = 13


## 4. Practical Task 1: Learn XOR with a Neural Network

### 4.1 Build data and inspect shapes
XOR is not linearly separable: no single straight line separates the two classes,
so we test whether a hidden layer with a nonlinearity can learn it.


In [3]:
import sys
sys.path.append("../src")

from xor_numpy_forward import build_xor_data, forward_pass

X, y = build_xor_data()
print("X shape:", X.shape, "-> axis 0 = 4 examples (batch), axis 1 = 2 input features (x1, x2)")
print("y shape:", y.shape, "-> one binary target per example")
print(X)
print(y)


X shape: (4, 2) -> axis 0 = 4 examples (batch), axis 1 = 2 input features (x1, x2)
y shape: (4,) -> one binary target per example
[[0. 0.]
 [0. 1.]
 [1. 0.]
 [1. 1.]]
[0. 1. 1. 0.]


### 4.2 NumPy forward pass (untrained) — understand shapes only

In [4]:
Z1, A1, logits, probs = forward_pass(X, seed=0)
print("Z1 shape:", Z1.shape, " (batch=4, hidden=8)")
print("A1 shape:", A1.shape, " (ReLU is elementwise, same shape as Z1)")
print("logits shape:", logits.shape, " (batch=4, output=1)")
print("probs shape:", probs.shape, " (sigmoid applied elementwise to logits)")
print(probs)


Z1 shape: (4, 8)  (batch=4, hidden=8)
A1 shape: (4, 8)  (ReLU is elementwise, same shape as Z1)
logits shape: (4, 1)  (batch=4, output=1)
probs shape: (4, 1)  (sigmoid applied elementwise to logits)
[[0.5       ]
 [0.50269265]
 [0.51650329]
 [0.52444958]]


### 4.3 Train the PyTorch XOR model
Required architecture: 2 → 8 (ReLU) → 1 logit. `BCEWithLogitsLoss` + `Adam`.

We return a **logit** (not a sigmoid) from the model because `BCEWithLogitsLoss`
combines the sigmoid and the binary cross-entropy loss internally in a single,
numerically stable operation. Applying sigmoid ourselves first and then using a
plain BCE loss is both redundant and less numerically stable.


In [6]:
from xor_torch import train_xor

result = train_xor(activation="relu", epochs=2000, lr=0.05, seed=42, verbose=True)


  [relu] epoch 500/2000  loss=0.0001
  [relu] epoch 1000/2000  loss=0.0000
  [relu] epoch 1500/2000  loss=0.0000
  [relu] epoch 2000/2000  loss=0.0000

  Final loss (relu): 0.0000
     input   target     prob   pred
  [0.0, 0.0]        0   0.0000      0
  [0.0, 1.0]        1   1.0000      1
  [1.0, 0.0]        1   1.0000      1
  [1.0, 1.0]        0   0.0000      0


## 5. Practical Task 2: Activation Function Experiment
Same architecture, data and optimiser — only the hidden activation changes.

In [7]:
from xor_torch import run_activation_experiment

activation_results = run_activation_experiment()



=== Training XOR with hidden activation = relu ===
  [relu] epoch 500/2000  loss=0.0001
  [relu] epoch 1000/2000  loss=0.0000
  [relu] epoch 1500/2000  loss=0.0000
  [relu] epoch 2000/2000  loss=0.0000

  Final loss (relu): 0.0000
     input   target     prob   pred
  [0.0, 0.0]        0   0.0000      0
  [0.0, 1.0]        1   1.0000      1
  [1.0, 0.0]        1   1.0000      1
  [1.0, 1.0]        0   0.0000      0

=== Training XOR with hidden activation = tanh ===
  [tanh] epoch 500/2000  loss=0.0002
  [tanh] epoch 1000/2000  loss=0.0001
  [tanh] epoch 1500/2000  loss=0.0000
  [tanh] epoch 2000/2000  loss=0.0000

  Final loss (tanh): 0.0000
     input   target     prob   pred
  [0.0, 0.0]        0   0.0000      0
  [0.0, 1.0]        1   1.0000      1
  [1.0, 0.0]        1   1.0000      1
  [1.0, 1.0]        0   0.0000      0

=== Training XOR with hidden activation = none ===
  [none] epoch 500/2000  loss=0.6931
  [none] epoch 1000/2000  loss=0.6931
  [none] epoch 1500/2000  loss=0.

Copy the printed table above into `reports/activation_experiment_results.md` with your own commentary.

## 6. Debugging Challenge

`src/debugging_challenge.py` contains:
- `broken_shape_example()` — a deliberately mismatched `nn.Linear` that raises a shape RuntimeError
- `fixed_shape_example()` — the corrected version
- `device_mismatch_example()` — shows a CPU/GPU mismatch (if a GPU is available) and the fix

**Individual competence check:** stacking only linear layers stays linear because composing
two linear functions is still linear — `W2(W1 x + b1) + b2 = (W2 W1) x + (W2 b1 + b2)`, which
has exactly the same form as a single linear layer with combined weights `W2 W1` and bias
`W2 b1 + b2`. No matter how many linear layers you stack, the whole network can still only
represent a straight decision boundary. A nonlinear hidden activation (ReLU, Tanh, ...) breaks
this collapsing — it lets each layer bend the representation, which is what allows the network
to represent non-linearly-separable patterns like XOR.


In [9]:
from debugging_challenge import fixed_shape_example, device_mismatch_example

# Uncomment the next two lines to see the real error message first:
# from debugging_challenge import broken_shape_example
# broken_shape_example()

fixed_shape_example()
device_mismatch_example()


Fixed shape example output shape: torch.Size([4, 1])
Detected device: cpu
After fix, output device: cpu


tensor([[ 0.9917],
        [-0.1565],
        [ 0.8459],
        [ 0.4352]], grad_fn=<AddmmBackward0>)

## 7. Practical Task 3: Fashion-MNIST MLP

Requires internet access the first time (to download the dataset) and `torchvision`.
`src/fashion_mnist_mlp.py` contains the full pipeline: load/split data, inspect a batch,
build `Flatten -> Linear(784,64) -> ReLU -> Linear(64,10)`, train 3 epochs with
`CrossEntropyLoss` + `Adam(lr=0.001)`, save the loss curve, and inspect 5 correct / 5
incorrect predictions.


In [10]:
from fashion_mnist_mlp import train

model, train_losses, val_losses = train()


ModuleNotFoundError: No module named 'torchvision'

The loss curve is saved to `../reports/loss_curve.png` and sample images to `../reports/sample_images.png`.

## 9. Reflection
See `reports/reflection.md` for the 5–8 line written reflection (what failed first, what
evidence helped fix it, what to test next).